# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Among FlyRank's tracked content pages, can page-level performance signals (position, CTR, engagement, historical trend) identify which pages are declining and worth a refresh — better than a simple hand-written rule?

**Decision this supports:** Prioritizing a content team's limited refresh capacity — which pages to review first (`snippet_fix`, `content_fix`, `monitor`) — rather than reviewing pages in an arbitrary or purely recency-based order.

In [ ]:
research_question = (
    "Can page-level performance signals identify declining content pages "
    "worth a refresh, better than a simple hand-written rule?"
)
decision_supported = (
    "Prioritizing a content team's limited refresh capacity across "
    "snippet_fix / content_fix / monitor actions."
)
print("Question:", research_question)
print("Decision:", decision_supported)

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` (Hugging Face) — `fact_content_daily_performance`, joined with `dim_content` and `dim_clients`.

**Window used:** March 2026 daily performance, aggregated to one row per (client, page).

**Eligibility filter:** ≥50 GSC impressions OR ≥10 GA4 sessions → **116,512 eligible pages**.

**Excluded:** pages below that threshold (too sparse to score reliably — a single visitor can swing engagement rate from 0% to 100%); raw client/domain identifiers (only `client_hash_id` / `content_hash_id` used throughout, per the public-safe rule).

In [1]:
# same data as the previous week assignments
!pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
).reset_index()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']
agg_df.loc[agg_df['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

eligible_mask = (agg_df['gsc_impressions'] >= 50) | (agg_df['ga4_sessions'] >= 10)
eligible_df = agg_df[eligible_mask].copy()

print(f"Total pages aggregated: {len(agg_df):,}")
print(f"Eligible pages (>=50 impressions OR >=10 sessions): {len(eligible_df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages aggregated: 331,437
Eligible pages (>=50 impressions OR >=10 sessions): 116,512


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.